# Insider Cluster Flows to CEX

This notebook ingests the per-unlock CSV exports, surfaces the wallets with the largest inbound SUI, builds lightweight clusters of addresses transacting with those wallets, and checks whether any of the clustered wallets route funds directly or within one hop to known CEX addresses.


In [26]:
from __future__ import annotations

import csv
import json
from collections import defaultdict, deque
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Set, Tuple

try:
    import pandas as pd  # type: ignore
except ImportError:
    pd = None

DATA_DIR = Path('..') / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

CLUSTER_MAX_DEPTH = 2
TOP_N = 10
MIN_INFLOW_SUI = 0
UNDIRECTED_GRAPH = True
# Provide CEX addresses in the next cell before running the analysis


In [27]:
# Populate with real CEX addresses before running deeper analysis.
# Format: label -> list of addresses
CEX_WALLETS = {
    'Binance': [
        '0x935029ca5219502a47ac9b69f556ccf6e2198b5e7815cf50f68846f723739cbd',
        '0xac5bceec1b789ff840d7d4e6ce4ce61c90d190a7f8c4f4ddf0bff6ee2413c33c',
    ],
    'Bybit': [
        '0x60dd01bc037e2c1ea2aaf02187701f9f4453ba323338d2f2f521957065b0984d',
    ],
    'Kraken': [
        '0x443cf42b0da43c230bff7a64e69ce25d24d65f49e7c9db6adecc0bd176dba79a',
    ],
    'Bitget': [
        '0xce7e1e38f996cdb2c4b78c1d187d23c1001d7b266f181498677672f9b1e24ea0',
        '0x3a1520d6be3a16f89655f542f6e5b050d3065ff67d7287ed8005259c4a0b36b7',
    ],
    'KuCoin': [
        '0x13c075644dad21bf10cb12447b825b535c24ddae428e27abc5858d1511e24c6d',
        '0xf3bfda04092a842d4c57c61f9d8fb99058792ab2622b2a0c0d83a6ea41de325f',
    ],
    'MEXC': [
        '0xf8b7b95d01ae79756fc3d1bc58675e59a17f07cdafe3d31ea57425adbec2d43f',
    ],
    'Gate': [
        '0x62f36b79d7ea8ae189491854edd9318b29c75346792177b230a95f333ffa53ad',
    ],
    'Crypto.com': [
        '0x7fa064f54d85caed3db8bdb5a3cb5d50b22a7f02f4954059df0e61f87e2bb7e9',
    ],
    'HTX': [
        '0x1f7b27844f2c4a0262b2c481f7ab956d10ace524c5a7b06c3742cfb8701db714',
    ],
    'Coinlist': [
        '0x7fb55e3e2f9ba133efa6223b3c66d15d4b979570c63cff6453041b45e8d6b1e9',
    ],
    'LBank': [
        '0x6cb04412036e910e5fc2237c69d154bb0f69e48a75b030f6919f044f485c8307',
    ],
    'Coinbase': [
        '0x0350925241ae8d4083fe4b1104e14dd28d2ffeced9d7575551c50c41b99995fb',
        '0x2e4706e435195185d69597fdb69bcab9e57a0fa297a14bbfbe70196017de12be',
    ],
    'OKX': [
        '0xab73ad38c63f83eda02182422b545395be1d3caeb54b5869159a9f70b678cd56',
    ],
    'Bitfinex_or_Gate': [
        '0x432875d9abcd8d3ed46e344b7dbeae952d014e8c26fc650e7503417a292cf43d',
    ],
}

ALL_CEX_ADDRESSES: Set[str] = set()
for addrs in CEX_WALLETS.values():
    ALL_CEX_ADDRESSES.update(addrs)


In [28]:
def load_unlock_csvs(data_dir: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    for csv_path in sorted(data_dir.glob('*.csv')):
        with csv_path.open('r', encoding='utf-8') as fh:
            reader = csv.DictReader(fh)
            for record in reader:
                try:
                    amount_sui = float(record.get('amount_sui', 0))
                except (TypeError, ValueError):
                    amount_sui = 0.0
                rows.append({
                    'event_label': record.get('event_label'),
                    'tx_digest': record.get('tx_digest'),
                    'checkpoint': record.get('checkpoint'),
                    'timestamp_ms': int(record.get('timestamp_ms') or 0),
                    'sender_address': record.get('sender_address'),
                    'recipient_address': record.get('recipient_address'),
                    'amount_sui': amount_sui,
                    'amount_mist': int(record.get('amount_mist') or 0),
                })
    return rows

RAW_ROWS = load_unlock_csvs(DATA_DIR)
if not RAW_ROWS:
    raise RuntimeError(f'No CSV files found under {DATA_DIR}. Generate unlock exports first.')
len(RAW_ROWS)


8484

In [29]:
def aggregate_inflows(rows: Sequence[Dict[str, Any]]) -> List[Dict[str, Any]]:
    inflow_totals: Dict[str, float] = defaultdict(float)
    tx_counts: Dict[str, int] = defaultdict(int)
    for row in rows:
        recipient = row['recipient_address']
        amount = row['amount_sui']
        if not recipient or amount <= 0:
            continue
        inflow_totals[recipient] += amount
        tx_counts[recipient] += 1
    combined = [
        {'recipient_address': addr, 'total_inflow_sui': inflow, 'tx_count': tx_counts[addr]}
        for addr, inflow in inflow_totals.items()
        if inflow >= MIN_INFLOW_SUI
    ]
    combined.sort(key=lambda item: item['total_inflow_sui'], reverse=True)
    return combined

INFLOWS = aggregate_inflows(RAW_ROWS)
if pd is not None:
    display(pd.DataFrame(INFLOWS[: TOP_N * 2]))
else:
    for row in INFLOWS[: TOP_N * 2]:
        print(row)
INFLOWS[:TOP_N]


,recipient_address,total_inflow_sui,tx_count
0,0x2b5b0a269f80e97b99abc0de05b74c18ee717c45f407...,468454.731348,107
1,0xf65ace09660db739c90cfdc09294053a69a12c2992ff...,248492.000000,5
2,0x6e75b7e1292f77920f77abb04ab0a235593e25ae2fc6...,175926.895846,1
3,0x86ce5faf1cd5e4e193f6e21717a3c546f208020aa4a1...,130114.101372,51
4,0xea4854c1569e2ef64acb3d4351d0569fcc09bdb9b67a...,98222.574343,37
5,0x8aa4c4f6446e761d6fbb01f3978b724cc78a963b07d0...,61992.360000,6
6,0x0ed1abb2912371136f5226375fd7b779b81093970e78...,50804.994056,37
7,0x436b1940c12a6dfcb2a1fa2197bb2b83d2e1fa7fe79a...,48994.519331,24
8,0x72f31c5612569fbbf269e2d9f9d2851c668b7a99c2ed...,47737.125082,35
9,0x8b302e06dd0bc3e09f1e009e1dc32a04f6b4b75ab550...,40376.937391,3


[{'recipient_address': '0x2b5b0a269f80e97b99abc0de05b74c18ee717c45f407d4a3159b97e47300d002',
  'total_inflow_sui': 468454.7313479591,
  'tx_count': 107},
 {'recipient_address': '0xf65ace09660db739c90cfdc09294053a69a12c2992ffba4cebe1eea15260868e',
  'total_inflow_sui': 248492.0,
  'tx_count': 5},
 {'recipient_address': '0x6e75b7e1292f77920f77abb04ab0a235593e25ae2fc662e42e7974aeebd23364',
  'total_inflow_sui': 175926.895846129,
  'tx_count': 1},
 {'recipient_address': '0x86ce5faf1cd5e4e193f6e21717a3c546f208020aa4a1c9b45a46a8e84535ab9d',
  'total_inflow_sui': 130114.101372177,
  'tx_count': 51},
 {'recipient_address': '0xea4854c1569e2ef64acb3d4351d0569fcc09bdb9b67afecdc4c3cdd9ff64cadf',
  'total_inflow_sui': 98222.57434271401,
  'tx_count': 37},
 {'recipient_address': '0x8aa4c4f6446e761d6fbb01f3978b724cc78a963b07d0a4dd7e420aa396abae24',
  'total_inflow_sui': 61992.36,
  'tx_count': 6},
 {'recipient_address': '0x0ed1abb2912371136f5226375fd7b779b81093970e78a965b0ec755ff707e004',
  'total_in

In [30]:
TOP_WALLETS = [entry['recipient_address'] for entry in INFLOWS[:TOP_N]]
if not TOP_WALLETS:
    raise RuntimeError('No recipient wallets met the inflow criteria.')
TOP_WALLETS


['0x2b5b0a269f80e97b99abc0de05b74c18ee717c45f407d4a3159b97e47300d002',
 '0xf65ace09660db739c90cfdc09294053a69a12c2992ffba4cebe1eea15260868e',
 '0x6e75b7e1292f77920f77abb04ab0a235593e25ae2fc662e42e7974aeebd23364',
 '0x86ce5faf1cd5e4e193f6e21717a3c546f208020aa4a1c9b45a46a8e84535ab9d',
 '0xea4854c1569e2ef64acb3d4351d0569fcc09bdb9b67afecdc4c3cdd9ff64cadf',
 '0x8aa4c4f6446e761d6fbb01f3978b724cc78a963b07d0a4dd7e420aa396abae24',
 '0x0ed1abb2912371136f5226375fd7b779b81093970e78a965b0ec755ff707e004',
 '0x436b1940c12a6dfcb2a1fa2197bb2b83d2e1fa7fe79a956402928f8cd5c31177',
 '0x72f31c5612569fbbf269e2d9f9d2851c668b7a99c2ed3cda531c63da6ddd0a4a',
 '0x8b302e06dd0bc3e09f1e009e1dc32a04f6b4b75ab5500f4137cb713de0a938de']

In [31]:
def build_adjacency(rows: Sequence[Dict[str, Any]], *, undirected: bool = True) -> Dict[str, Set[str]]:
    adjacency: Dict[str, Set[str]] = defaultdict(set)
    for row in rows:
        sender = row['sender_address']
        recipient = row['recipient_address']
        if not sender or not recipient:
            continue
        adjacency[sender].add(recipient)
        if undirected:
            adjacency[recipient].add(sender)
    return adjacency

ADJACENCY = build_adjacency(RAW_ROWS, undirected=UNDIRECTED_GRAPH)
len(ADJACENCY)


399

In [32]:
def bfs_cluster(seed: str, adjacency: Dict[str, Set[str]], max_depth: int) -> Set[str]:
    visited: Set[str] = {seed}
    frontier: deque[Tuple[str, int]] = deque([(seed, 0)])
    while frontier:
        node, depth = frontier.popleft()
        if depth >= max_depth:
            continue
        for neighbor in adjacency.get(node, set()):
            if neighbor in visited:
                continue
            visited.add(neighbor)
            frontier.append((neighbor, depth + 1))
    return visited

CLUSTERS: Dict[str, Set[str]] = {}
for wallet in TOP_WALLETS:
    CLUSTERS[wallet] = bfs_cluster(wallet, ADJACENCY, CLUSTER_MAX_DEPTH)
{wallet: len(nodes) for wallet, nodes in CLUSTERS.items()}


{'0x2b5b0a269f80e97b99abc0de05b74c18ee717c45f407d4a3159b97e47300d002': 1,
 '0xf65ace09660db739c90cfdc09294053a69a12c2992ffba4cebe1eea15260868e': 5,
 '0x6e75b7e1292f77920f77abb04ab0a235593e25ae2fc662e42e7974aeebd23364': 5,
 '0x86ce5faf1cd5e4e193f6e21717a3c546f208020aa4a1c9b45a46a8e84535ab9d': 5,
 '0xea4854c1569e2ef64acb3d4351d0569fcc09bdb9b67afecdc4c3cdd9ff64cadf': 5,
 '0x8aa4c4f6446e761d6fbb01f3978b724cc78a963b07d0a4dd7e420aa396abae24': 77,
 '0x0ed1abb2912371136f5226375fd7b779b81093970e78a965b0ec755ff707e004': 5,
 '0x436b1940c12a6dfcb2a1fa2197bb2b83d2e1fa7fe79a956402928f8cd5c31177': 4,
 '0x72f31c5612569fbbf269e2d9f9d2851c668b7a99c2ed3cda531c63da6ddd0a4a': 5,
 '0x8b302e06dd0bc3e09f1e009e1dc32a04f6b4b75ab5500f4137cb713de0a938de': 6}

In [33]:
def index_transactions(rows: Sequence[Dict[str, Any]]) -> Tuple[Dict[str, List[Dict[str, Any]]], Dict[str, List[Dict[str, Any]]]]:
    by_sender: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    by_recipient: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for row in rows:
        sender = row['sender_address']
        recipient = row['recipient_address']
        if sender:
            by_sender[sender].append(row)
        if recipient:
            by_recipient[recipient].append(row)
    return by_sender, by_recipient

TX_BY_SENDER, TX_BY_RECIPIENT = index_transactions(RAW_ROWS)
len(TX_BY_SENDER), len(TX_BY_RECIPIENT)


(329, 330)

In [34]:
def analyze_cex_exposure(cluster: Set[str]) -> Dict[str, Any]:
    direct_hits: List[Dict[str, Any]] = []
    one_hop_hits: List[Dict[str, Any]] = []
    for sender in cluster:
        for tx in TX_BY_SENDER.get(sender, []):
            recipient = tx['recipient_address']
            amount = tx['amount_sui']
            if not recipient or amount <= 0:
                continue
            if recipient in ALL_CEX_ADDRESSES:
                direct_hits.append({
                    'sender': sender,
                    'recipient': recipient,
                    'amount_sui': amount,
                    'tx_digest': tx['tx_digest'],
                    'event_label': tx['event_label'],
                })
            else:
                for downstream in TX_BY_SENDER.get(recipient, []):
                    downstream_recipient = downstream['recipient_address']
                    if downstream_recipient in ALL_CEX_ADDRESSES and downstream['amount_sui'] > 0:
                        one_hop_hits.append({
                            'sender': sender,
                            'intermediate': recipient,
                            'cex_recipient': downstream_recipient,
                            'amount_sui': downstream['amount_sui'],
                            'initial_tx_digest': tx['tx_digest'],
                            'cex_tx_digest': downstream['tx_digest'],
                            'event_label': tx['event_label'],
                        })
    total_direct = sum(hit['amount_sui'] for hit in direct_hits)
    total_one_hop = sum(hit['amount_sui'] for hit in one_hop_hits)
    return {
        'direct_hits': direct_hits,
        'one_hop_hits': one_hop_hits,
        'total_direct_sui': total_direct,
        'total_one_hop_sui': total_one_hop,
    }


In [35]:
SUMMARY_ROWS: List[Dict[str, Any]] = []
DETAILED_RESULTS: Dict[str, Dict[str, Any]] = {}
for seed_wallet in TOP_WALLETS:
    cluster = CLUSTERS.get(seed_wallet, {seed_wallet})
    exposure = analyze_cex_exposure(cluster)
    SUMMARY_ROWS.append({
        'seed_wallet': seed_wallet,
        'cluster_size': len(cluster),
        'total_direct_sui': exposure['total_direct_sui'],
        'total_one_hop_sui': exposure['total_one_hop_sui'],
    })
    DETAILED_RESULTS[seed_wallet] = exposure
if pd is not None:
    display(pd.DataFrame(SUMMARY_ROWS))
else:
    for row in SUMMARY_ROWS:
        print(row)
SUMMARY_ROWS


,seed_wallet,cluster_size,total_direct_sui,total_one_hop_sui
0,0x2b5b0a269f80e97b99abc0de05b74c18ee717c45f407...,1,0.000000,0.000000
1,0xf65ace09660db739c90cfdc09294053a69a12c2992ff...,5,0.000000,0.000000
2,0x6e75b7e1292f77920f77abb04ab0a235593e25ae2fc6...,5,28196.716521,0.000000
3,0x86ce5faf1cd5e4e193f6e21717a3c546f208020aa4a1...,5,0.000000,0.000000
4,0xea4854c1569e2ef64acb3d4351d0569fcc09bdb9b67a...,5,0.000000,0.000000
5,0x8aa4c4f6446e761d6fbb01f3978b724cc78a963b07d0...,77,75960.366838,129487.219587
6,0x0ed1abb2912371136f5226375fd7b779b81093970e78...,5,0.000000,0.000000
7,0x436b1940c12a6dfcb2a1fa2197bb2b83d2e1fa7fe79a...,4,0.000000,0.000000
8,0x72f31c5612569fbbf269e2d9f9d2851c668b7a99c2ed...,5,0.000000,0.000000
9,0x8b302e06dd0bc3e09f1e009e1dc32a04f6b4b75ab550...,6,14302.939239,99709.220395


[{'seed_wallet': '0x2b5b0a269f80e97b99abc0de05b74c18ee717c45f407d4a3159b97e47300d002',
  'cluster_size': 1,
  'total_direct_sui': 0,
  'total_one_hop_sui': 0},
 {'seed_wallet': '0xf65ace09660db739c90cfdc09294053a69a12c2992ffba4cebe1eea15260868e',
  'cluster_size': 5,
  'total_direct_sui': 0,
  'total_one_hop_sui': 0},
 {'seed_wallet': '0x6e75b7e1292f77920f77abb04ab0a235593e25ae2fc662e42e7974aeebd23364',
  'cluster_size': 5,
  'total_direct_sui': 28196.71652096,
  'total_one_hop_sui': 0},
 {'seed_wallet': '0x86ce5faf1cd5e4e193f6e21717a3c546f208020aa4a1c9b45a46a8e84535ab9d',
  'cluster_size': 5,
  'total_direct_sui': 0,
  'total_one_hop_sui': 0},
 {'seed_wallet': '0xea4854c1569e2ef64acb3d4351d0569fcc09bdb9b67afecdc4c3cdd9ff64cadf',
  'cluster_size': 5,
  'total_direct_sui': 0,
  'total_one_hop_sui': 0},
 {'seed_wallet': '0x8aa4c4f6446e761d6fbb01f3978b724cc78a963b07d0a4dd7e420aa396abae24',
  'cluster_size': 77,
  'total_direct_sui': 75960.36683773997,
  'total_one_hop_sui': 129487.2195868

In [36]:
def pretty_print_exposure(seed_wallet: str) -> None:
    exposure = DETAILED_RESULTS.get(seed_wallet)
    if exposure is None:
        print(f'No exposure data for {seed_wallet}')
        return
    print(f'=== {seed_wallet} ===')
    print(f"Direct to CEX total: {exposure['total_direct_sui']:.2f} SUI across {len(exposure['direct_hits'])} tx")
    for hit in exposure['direct_hits']:
        print('  ->', hit)
    print(f"One-hop to CEX total: {exposure['total_one_hop_sui']:.2f} SUI across {len(exposure['one_hop_hits'])} paths")
    for hit in exposure['one_hop_hits']:
        print('  ->', hit)

# Example usage:
# pretty_print_exposure(TOP_WALLETS[0])
